# Denoising via Burst — Pipeline Test
Run each cell in order. Works with synthetic data or real CR3 files.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from pipeline.raw_loader import make_synthetic_burst, load_raw_bayer
from pipeline.registration import register_burst, select_reference_frame, estimate_sharpness
from pipeline.denoising import denoise_burst, estimate_noise_sigma, compute_psnr, compute_ssim, ALL_METHODS
from pipeline.isp import run_isp, detect_bayer_pattern

print('All imports OK')

## 1. Load frames
Choose **one** of the two cells below.

In [ ]:
# ── Option A: synthetic burst (no camera needed) ──────────────────────────────
frames = make_synthetic_burst(height=512, width=512, n_frames=8, noise_sigma=0.07)
bayer_pattern = 'RGGB'   # synthetic data is always RGGB
print(f'Loaded {len(frames)} synthetic frames, shape={frames[0].shape}')

In [ ]:
# ── Option B: real CR3 burst ──────────────────────────────────────────────────
# Put your CR3 files in data/ and set the folder path below
# Then run this cell manually (skipped during auto-execute)
import os
from pathlib import Path

burst_dir = Path('data/your_scene')   # <-- change this
raw_exts  = {'.cr3', '.cr2', '.tiff', '.tif', '.png'}
files     = sorted(p for p in burst_dir.iterdir() if p.suffix.lower() in raw_exts)

frames = []
bayer_pattern = 'auto'
for f in files:
    bayer, pattern = load_raw_bayer(f)
    frames.append(bayer)
    if bayer_pattern == 'auto' and pattern not in ('unknown', ''):
        bayer_pattern = pattern
    print(f'  {f.name}  pattern={pattern}')

print(f'\nLoaded {len(frames)} frames, Bayer pattern: {bayer_pattern}')


## 2. Inspect raw frames

In [ ]:
fig, axes = plt.subplots(1, min(4, len(frames)), figsize=(14, 3.5))
if len(frames) == 1:
    axes = [axes]
for i, ax in enumerate(axes):
    ax.imshow(frames[i], cmap='gray', vmin=0, vmax=1)
    sharpness = estimate_sharpness(frames[i])
    ax.set_title(f'Frame {i}\nsharpness={sharpness:.0f}', fontsize=9)
    ax.axis('off')
plt.suptitle('Raw Bayer frames (first 4)', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

print(f'Noise sigma estimate: {estimate_noise_sigma(frames[0]):.4f}')

## 3. Registration

In [ ]:
REG_METHOD = 'ecc'   # 'ecc' | 'feature' | 'phase'

ref_idx = select_reference_frame(frames)
print(f'Reference frame: {ref_idx} (sharpest)')

aligned, transforms = register_burst(frames, reference_idx=ref_idx, method=REG_METHOD)
print(f'Registration done ({REG_METHOD}). {len(aligned)} frames aligned.')

# Show alignment quality: pixel difference vs reference
fig, axes = plt.subplots(1, min(4, len(frames)), figsize=(14, 3.5))
if len(aligned) == 1:
    axes = [axes]
for i, ax in enumerate(axes[:len(axes)]):
    diff = np.abs(aligned[i] - aligned[ref_idx])
    ax.imshow(diff, cmap='hot', vmin=0, vmax=0.1)
    ax.set_title(f'|frame{i} - ref|\nmax={diff.max():.3f}', fontsize=9)
    ax.axis('off')
plt.suptitle('Registration residuals (darker = better aligned)', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

## 4. Denoising — compare all methods

In [ ]:
# Run all 4 denoising methods
results = {}
metrics = {}

for method in ALL_METHODS:
    denoised = denoise_burst(aligned, method=method)
    rgb      = run_isp(denoised, bayer_pattern=bayer_pattern)
    psnr     = compute_psnr(aligned[ref_idx], denoised)
    ssim     = compute_ssim(aligned[ref_idx], denoised)
    results[method] = rgb
    metrics[method] = {'psnr': psnr, 'ssim': ssim}
    print(f'  [{method:10s}]  PSNR={psnr:.1f} dB   SSIM={ssim:.3f}')

# Also run ISP on the raw reference (noisy baseline)
noisy_rgb = run_isp(frames[ref_idx], bayer_pattern=bayer_pattern)

In [ ]:
# Visual comparison
n_cols = 1 + len(ALL_METHODS)
fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))

axes[0].imshow(noisy_rgb)
axes[0].set_title('Noisy\n(single frame)', fontsize=9)
axes[0].axis('off')

for ax, method in zip(axes[1:], ALL_METHODS):
    ax.imshow(results[method])
    m = metrics[method]
    ax.set_title(f'{method}\nPSNR={m["psnr"]:.1f} dB  SSIM={m["ssim"]:.3f}', fontsize=9)
    ax.axis('off')

plt.suptitle('Denoising method comparison', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 5. Metrics bar chart

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))

methods = list(metrics.keys())
psnrs   = [metrics[m]['psnr'] for m in methods]
ssims   = [metrics[m]['ssim'] for m in methods]
colors  = ['#a78bfa', '#4f8ef7', '#f0a84e', '#7ec8a4']

bars1 = ax1.bar(methods, psnrs, color=colors, edgecolor='none')
ax1.set_ylabel('PSNR (dB)')
ax1.set_title('PSNR — higher is better')
for bar, val in zip(bars1, psnrs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             f'{val:.1f}', ha='center', va='bottom', fontsize=8)

bars2 = ax2.bar(methods, ssims, color=colors, edgecolor='none')
ax2.set_ylabel('SSIM')
ax2.set_title('SSIM — higher is better')
for bar, val in zip(bars2, ssims):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{val:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

## 6. ISP stage-by-stage walkthrough
Pick one denoising method and visualise every ISP stage.

In [ ]:
import os, cv2
from pathlib import Path

DENOISE_METHOD = 'weighted'   # change to any of ALL_METHODS

stages_dir = Path('results/notebook_stages')
denoised   = denoise_burst(aligned, method=DENOISE_METHOD)

run_isp(
    denoised,
    bayer_pattern=bayer_pattern,
    save_stages=True,
    stages_dir=stages_dir,
    scene_name=DENOISE_METHOD,
)

# Load and display all stage images
stage_files = sorted(stages_dir.glob(f'{DENOISE_METHOD}_stage*.png'))
n = len(stage_files)
fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(16, 7))
axes = axes.flatten()

for ax, fpath in zip(axes, stage_files):
    img = cv2.cvtColor(cv2.imread(str(fpath)), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(fpath.stem.split('_stage')[1], fontsize=8)
    ax.axis('off')

for ax in axes[n:]:
    ax.axis('off')

plt.suptitle(f'ISP stages — {DENOISE_METHOD} denoising', fontsize=12)
plt.tight_layout()
plt.show()

## 7. Bayer pattern check

In [ ]:
detected = detect_bayer_pattern(frames[0])
print(f'Auto-detected pattern : {detected}')
print(f'Pattern used by loader: {bayer_pattern}')

# Show all 4 demosaic interpretations side by side
from pipeline.isp import demosaic
patterns = ['RGGB', 'BGGR', 'GRBG', 'GBRG']

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for ax, pat in zip(axes, patterns):
    rgb_test = demosaic(frames[ref_idx], method='bilinear', pattern=pat)
    rgb_u8   = (np.clip(rgb_test, 0, 1) * 255).astype(np.uint8)
    ax.imshow(rgb_u8)
    label = f'{pat}'
    if pat == detected:
        label += ' ← auto-detected'
    if pat == bayer_pattern:
        label += ' ← used'
    ax.set_title(label, fontsize=9)
    ax.axis('off')

plt.suptitle('Same frame demosaiced with all 4 Bayer patterns\n(wrong pattern = colour cast or green tint)', fontsize=10)
plt.tight_layout()
plt.show()

## 8. Save best result

In [ ]:
import cv2
from pathlib import Path

BEST_METHOD  = 'nlm'          # change to whichever looked best above
OUTPUT_PATH  = Path('results/best_result.jpg')
JPEG_QUALITY = 95

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

denoised_best = denoise_burst(aligned, method=BEST_METHOD)
rgb_best = run_isp(
    denoised_best,
    bayer_pattern=bayer_pattern,
    jpeg_quality=JPEG_QUALITY,
    output_path=str(OUTPUT_PATH),
)

print(f'Saved -> {OUTPUT_PATH}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ax1.imshow(noisy_rgb)
ax1.set_title('Before (single noisy frame)', fontsize=10)
ax1.axis('off')
ax2.imshow(rgb_best)
ax2.set_title(f'After ({BEST_METHOD} denoising)', fontsize=10)
ax2.axis('off')
plt.tight_layout()
plt.show()